# Tutorial: Using `DataSet` as a Context Manager

`ep.dataset.DataSet` (and its concrete subclasses `GFZDataSet`/`PRBEMDataSet`) can be used exactly as before, by simply instantiating it, or as a context manager using the `with` statement:

```python
with ep.dataset.DataSet(...) as ds:
    ...
```

This tutorial shows both usage styles and explains why the context manager is useful.


## Setup

We create a small mock dataset and save it with `MonthlyRBStrategy` using the default `.nc` (NetCDF) format. `DataSet` loads `.nc` files lazily via `xarray`, which is exactly the case where the context manager is useful, as explained further below.


In [1]:
from datetime import datetime, timezone
import logging

import numpy as np
from astropy import units as u

import el_paso as ep
from el_paso.dataset import DataSet
from el_paso.dataset.utils import python2matlab

ep.setup_logging()

logger = logging.getLogger(__name__)

time_size = 20
start_time = datetime(2013, 1, 1, tzinfo=timezone.utc)
datetimes = [start_time + i * np.timedelta64(6000, "s") for i in range(time_size)]
end_time = datetimes[-1]

variables = {
    "Epoch": ep.Variable(original_unit=ep.units.datenum, data=np.array([python2matlab(dt) for dt in datetimes])),
    "FEDU": ep.Variable(
        original_unit=(u.cm**2 * u.s * u.sr * u.keV) ** (-1),
        data=np.random.default_rng(0).random((time_size, 3, 4)),
    ),
    "Alpha": ep.Variable(original_unit=u.deg, data=np.tile([10.0, 30.0, 60.0, 90.0], (time_size, 1))),
    "Energy_FEDU": ep.Variable(original_unit=u.MeV, data=np.tile([0.5, 1.0, 2.0], (time_size, 1))),
}
for variable in variables.values():
    variable.metadata.source_files = ["mocked_input.cdf"]

strategy = ep.saving_strategies.MonthlyRBStrategy(
    base_data_path="./context_manager_example",
    mission="MOCK",
    satellite="sat",
    instrument="inst",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)

ep.save(variables, strategy, start_time=start_time, end_time=end_time, time_var=variables["Epoch"])


[WARNING ] 2026-07-26 01:33:15 - el_paso.saving_strategy:81 - Could not find target variable(s) Alpha_Eq, B_Calc, B_Eq, InvK, InvMu, L_m, L_star, MLT, PSD, Position, R_Eq!
[INFO    ] 2026-07-26 01:33:15 - el_paso.saving_strategy:183 - Saving file: /Users/jhawarji/EL_PASO/tutorials/context_manager_example/MOCK/sat/sat_inst_20130101to20130131_T89.nc
[INFO    ] 2026-07-26 01:33:15 - el_paso.save:41 - save finished in 0.020 seconds


## Using `DataSet` directly

This is the usage you are already familiar with from the other tutorials: simply instantiate `DataSet` and access variables. Nothing changes here, this keeps working exactly as before.


In [2]:
legacy_ds = DataSet(strategy, start_time, end_time)
legacy_ds.load("Flux")
legacy_ds.Flux.shape


[INFO    ] 2026-07-26 01:33:15 - el_paso.dataset.dataset:401 - Loading context_manager_example/MOCK/sat/sat_inst_20130101to20130131_T89.nc


(20, 3, 4)

## Using `DataSet` as a context manager

Alternatively, `DataSet` implements `__enter__`/`__exit__`, so it can be used with the `with` statement. On exit, any file handles that were opened while lazily loading `.nc` files are closed automatically.


In [3]:
with DataSet(strategy, start_time, end_time) as ds:
    ds.load("Flux")
    logger.info(f"Inside the `with` block: {ds.Flux.shape}")

# Already-loaded variables remain accessible after the `with` block exits,
# only the underlying open file handles are released.
logger.info(f"After the `with` block: {ds.Flux.shape}")


[INFO    ] 2026-07-26 01:33:15 - el_paso.dataset.dataset:401 - Loading context_manager_example/MOCK/sat/sat_inst_20130101to20130131_T89.nc
[INFO    ] 2026-07-26 01:33:15 - __main__:3 - Inside the `with` block: (20, 3, 4)
[INFO    ] 2026-07-26 01:33:15 - __main__:7 - After the `with` block: (20, 3, 4)


### Why does this matter?

`DataSet` loads `.nc` files lazily through `xarray`. Opening a file is cheap (only structure/metadata is read); the first time a variable is actually accessed, its data is read from disk and then cached in memory, so repeated access to the *same* variable never touches the file again.

You might reasonably ask: if xarray already caches materialized data and even caps how many files it keeps open concurrently (128 by default, evicting the least-recently-used one automatically), does closing explicitly matter at all? For reading, not much, if you access a variable on a dataset whose file handle happens to be closed (explicitly or because xarray's cache evicted it), it is transparently reopened for you.

Where it does matter is when you need *this exact file* to be free for something else right now, most commonly: overwriting it. The next cell reproduces that failure directly.


In [4]:
try:
    ep.save(variables, strategy, start_time=start_time, end_time=end_time, time_var=variables["Epoch"])
except PermissionError as exc:
    logger.error(f"Overwrite failed while `legacy_ds` still has the file open:\n  {exc}")


[WARNING ] 2026-07-26 01:33:15 - el_paso.saving_strategy:81 - Could not find target variable(s) Alpha_Eq, B_Calc, B_Eq, InvK, InvMu, L_m, L_star, MLT, PSD, Position, R_Eq!
[INFO    ] 2026-07-26 01:33:15 - el_paso.saving_strategy:183 - Saving file: /Users/jhawarji/EL_PASO/tutorials/context_manager_example/MOCK/sat/sat_inst_20130101to20130131_T89.nc
[ERROR   ] 2026-07-26 01:33:15 - __main__:4 - Overwrite failed while `legacy_ds` still has the file open:
  [Errno 13] Permission denied: 'context_manager_example/MOCK/sat/sat_inst_20130101to20130131_T89.nc'


Closing `legacy_ds` releases the file, and the same overwrite now succeeds:


In [5]:
legacy_ds.close()

ep.save(variables, strategy, start_time=start_time, end_time=end_time, time_var=variables["Epoch"])
logger.info("Overwrite succeeded after legacy_ds.close().")


[WARNING ] 2026-07-26 01:33:15 - el_paso.saving_strategy:81 - Could not find target variable(s) Alpha_Eq, B_Calc, B_Eq, InvK, InvMu, L_m, L_star, MLT, PSD, Position, R_Eq!
[INFO    ] 2026-07-26 01:33:15 - el_paso.saving_strategy:183 - Saving file: /Users/jhawarji/EL_PASO/tutorials/context_manager_example/MOCK/sat/sat_inst_20130101to20130131_T89.nc
[INFO    ] 2026-07-26 01:33:15 - el_paso.save:3 - save finished in 0.021 seconds
[INFO    ] 2026-07-26 01:33:15 - __main__:4 - Overwrite succeeded after legacy_ds.close().


There is one more subtlety worth knowing about. Loading `"Flux"` also cached every *other* variable found in the same file (e.g. `alpha_local`), but those were only cached lazily, their data was never actually read from disk. `close()` also materializes every such variable up front, precisely so that touching one later can never reopen the file behind your back:


In [6]:
_ = legacy_ds.alpha_local  # already a plain array, materialized by the close() call above -- no disk access here
legacy_ds.close()  # still a no-op

ep.save(variables, strategy, start_time=start_time, end_time=end_time, time_var=variables["Epoch"])
logger.info("Overwrite still succeeds, even after touching a variable that was cached but never explicitly accessed before.")


[WARNING ] 2026-07-26 01:33:15 - el_paso.saving_strategy:81 - Could not find target variable(s) Alpha_Eq, B_Calc, B_Eq, InvK, InvMu, L_m, L_star, MLT, PSD, Position, R_Eq!
[INFO    ] 2026-07-26 01:33:15 - el_paso.saving_strategy:183 - Saving file: /Users/jhawarji/EL_PASO/tutorials/context_manager_example/MOCK/sat/sat_inst_20130101to20130131_T89.nc
[INFO    ] 2026-07-26 01:33:15 - el_paso.save:4 - save finished in 0.011 seconds
[INFO    ] 2026-07-26 01:33:15 - __main__:5 - Overwrite still succeeds, even after touching a variable that was cached but never explicitly accessed before.


### Exception safety

Just like closing a file, `__exit__` runs even if an exception is raised inside the `with` block, so open file handles are never leaked because of an error while processing.


In [7]:
try:
    with DataSet(strategy, start_time, end_time, verbose=False) as ds:
        ds.load("Flux")
        raise RuntimeError("something went wrong while processing")
except RuntimeError as exc:
    logger.error(f"Caught: {exc}")
    logger.warning("File handles opened above were still closed by __exit__.")


[ERROR   ] 2026-07-26 01:33:15 - __main__:6 - Caught: something went wrong while processing
[WARNING ] 2026-07-26 01:33:15 - __main__:7 - File handles opened above were still closed by __exit__.


### Idiomatic loop usage

Beyond the overwrite case above, `with` is also the natural pattern when creating many `DataSet` instances in a loop (e.g. one per satellite or time range), since each one is released as soon as you are done with it, rather than accumulating until something else cleans them up:


In [8]:
for satellite in ["sat"]:
    with DataSet(strategy, start_time, end_time, verbose=False) as ds:
        ds.load("Flux")
        logger.info(f"{satellite}: Flux mean = {ds.Flux.mean():.3f}")
    # the file handle opened for this satellite is released here, before the next iteration


[INFO    ] 2026-07-26 01:33:15 - __main__:4 - sat: Flux mean = 0.532


## Both styles are fully supported

You can pick whichever style fits your use case, `DataSet` does not force either one on you:

- **Direct usage** (`ds = DataSet(...)`): simplest for quick, interactive exploration. If you never call `close()`, the underlying `__del__` closes any remaining open file handles once the object is garbage-collected, same as before.
- **Context manager usage** (`with DataSet(...) as ds:`): recommended whenever you need a file released deterministically, such as right before overwriting it, as shown above, or when creating many `DataSet` instances in a loop.

You can also call `close()` directly at any point, on the specific object that holds the file open; it is safe to call multiple times.


In [9]:
legacy_ds.close()
legacy_ds.close()  # calling close() again is a no-op